String Separator - PV Toolestown
=
This notebook is the Toolestown copy of *Separador de Strings - Reask*. It reads the electromechanical drawing of PV Toolestown, numbers the strings of each inverter (`STR`), and groups neighbouring strings of the same table row into trios (with a pair or a single string at the row end). It writes an AutoCAD script (`.scr`) that inserts one `EMF422-1_string_1/2/3` block per string, tagged with `UFV`, `PST`, `INV`, `STR` and `GRP`.

**Differences from the Reask version:** the block names are `EMF422-2_...` and there is no anonymous string block. There are no row-count checks and no `teste3.xlsx` export, and more debug prints are active. The script still inserts the `EMF422-1_string_N` blocks (the Reask names).

**Workflow overview:**
1. Convert the DWG to DXF and extract the model space entities into memory (this notebook doesn't use DuckDB).
2. Select the string blocks and the inverter blocks.
3. Build the `UFV.PST.INV` key (`GRP`) on both and join each string to its inverter.
4. Decide on which side of its strings each inverter is (`E or W`), and number the strings of each inverter top-to-bottom / left-to-right.
5. Cut each table row into trios, starting from the end away from the inverter, so the leftover pair/single ends up on the inverter side.
6. Build the insert commands and write `Separador de Strings - Toolestown.scr`.

**Tag hierarchy:** `UFV` → `PST` → `INV` → `STR`, and `GRP` = `UFV.PST.INV`.

**DWG prerequisites** (the code assumes all of these, and it doesn't check them):
- **File:** `PV Toolestown - Eletromecânico.dwg` sits in `dwg_folder`. The name is hardcoded in three places in the first code cell. The ODA File Converter 27.1.0 must be installed at the path set in `DWG_to_DuckDB_ETL.py`.
- **Model space only:** strings and inverters are block references (`INSERT`) directly in model space.
- **Attributes already filled:** every string block and every inverter block already carries `UFV`, `PST` and `INV` attributes, which this notebook reuses as they are (it only numbers `STR`).
  - `PST` must already be 2 digits (`01`); `UFV` and the string `INV` are zero-padded by the code.
  - The inverter `INV` must be numeric text (it is cast to `Int64`).
  - Each `UFV.PST.INV` must appear on only one inverter block, or the merge duplicates that inverter's strings.
- **Other attributes:** the drawing must contain blocks with `EFX`, `STR`, `VBT`, `QBT`, `DSJ` and `PVM` attributes, because the code drops those columns by name (`drop` raises a `KeyError` if one is missing).
- **Layout:** strings are placed in horizontal rows: the strings of a row share the same Y (within 0.1), and neighbours in a row are no more than 12 m apart (`max_distance`). The block rotation is ignored: the new blocks are inserted at 0° (and 180° for the second block of a pair).
  - The Toolestown drawing does contain rotated blocks (rotation up to 286.6° in the inspection output), so check that the string rows really are horizontal.
- **Layers:** each new block goes on the layer of the string it replaces, so no extra layers are needed.
- **Block names** (exact, case-sensitive `==` match): strings `EMF422-2_string_tilt_15°`, inverters `EMF422-2_inversor`.

**AutoCAD prerequisites for the generated script:**
- The blocks `EMF422-1_string_1`, `EMF422-1_string_2` and `EMF422-1_string_3` (the Reask names, hardcoded in `generate_script`) are defined in the Toolestown drawing, with *Scale uniformly* off (the script answers X and Y scale, `1 1`).
- Their attributes are filled positionally in this order: `UFV`, `PST`, `INV`, an empty value (the `EFX` slot), `STR`, `GRP`.
- The offsets `5.86` (pairs) and `2.3204` (trios) are tied to the geometry of these blocks and to the 11.72 m string pitch.
- Run the `.scr` with `ATTREQ = 1` and `ATTDIA = 0`. The script only inserts: it doesn't erase the original string blocks.

Extracting the AutoCAD data
=
- Defines `dwg_folder` (source DWG), `temp_folder` (working copy) and `output_folder` (where the DXF is written, the same as `dwg_folder`).
- Copies the DWG into `temp_folder` so the ODA File Converter only processes that single file.
- `convert_dwg_to_dxf` (from `DWG_to_DuckDB_ETL.py`) converts it to DXF (ACAD2018).
- `extract_data_from_dxf` reads every model space entity with `ezdxf` into a list of dicts. For block references it keeps the block name, insertion point (`X`, `Y`), rotation, color and every attribute tag/value. Nothing is loaded into DuckDB here.
- Deletes the DXF and the copied DWG at the end.
- The drawing name is repeated in three paths: change all three when switching drawings. `math` is imported but not used.

In [55]:
import pandas as pd

from DWG_to_DuckDB_ETL import *
import os
import shutil
import numpy as np
import math

dwg_folder = r"C:\Users\Usuario\Ecotechnee\Office - Documentos\Projetos\422 - Zagope - UFV Reask e Tooletown\2 - Toolestown Solar\01 - Projetos\01 - Em Confecção\02 - Edição\01 - Layout"
temp_folder = r"C:\Users\Usuario\Ecotechnee\Office - Documentos\Projetos\422 - Zagope - UFV Reask e Tooletown\2 - Toolestown Solar\01 - Projetos\01 - Em Confecção\02 - Edição\01 - Layout\temp"
output_folder = dwg_folder

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Toolestown - Eletromecânico.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Toolestown - Eletromecânico.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Toolestown - Eletromecânico.dwg")

data = extract_data_from_dxf(dxf_file)
print("Data Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Data Loaded


Inspecting the entities
=
Puts the extracted entities in a DataFrame (`ELM_df`) and prints `describe()`, `info()` and the count of each block name. Use the counts to confirm the block names used in the masks of the next steps.

In [56]:
ELM_df = pd.DataFrame(data)

print(ELM_df.describe())
print(ELM_df.info())
print(ELM_df['name'].value_counts())

                   X              Y     rotation
count    4643.000000    4643.000000  4643.000000
mean   693759.743063  735293.876534    87.739394
std      1154.692166     681.912543    90.024161
min    692171.852937  734013.012016     0.000000
25%    692626.137864  734734.073055     0.000000
50%    694336.803755  735138.650535     0.000000
75%    694818.117051  735903.400560   180.000000
max    695492.483506  736344.394819   286.593056
<class 'pandas.DataFrame'>
RangeIndex: 4643 entries, 0 to 4642
Data columns (total 17 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   type      4643 non-null   str    
 1   layer     4643 non-null   str    
 2   handle    4643 non-null   str    
 3   name      4643 non-null   str    
 4   X         4643 non-null   float64
 5   Y         4643 non-null   float64
 6   rotation  4643 non-null   float64
 7   UFV       4643 non-null   str    
 8   PST       4643 non-null   str    
 9   INV       4632 non-null   str

String blocks
=
- Keeps the blocks named exactly `EMF422-2_string_tilt_15°`.
- Renames `X`/`Y` to `Position_X`/`Position_Y` and drops the columns not needed for strings (`rotation`, `type`, `handle`, `QBT`, `DSJ`, `PVM`, `VBT`).
- Writes a debug copy to `C:\Users\Usuario\Desktop\teste2.xlsx`.

In [57]:
struct_mask = (ELM_df['name'] == "EMF422-2_string_tilt_15°")

struct_df = ELM_df[struct_mask].copy()
struct_df.rename(columns={'X': 'Position_X', 'Y': 'Position_Y'}, inplace=True)
struct_df.drop(columns=['rotation', 'type', 'handle', 'QBT', 'DSJ', 'PVM', 'VBT'], inplace=True)
struct_df.to_excel(r"C:\Users\Usuario\Desktop\teste2.xlsx")
print(struct_df)
print(struct_df.describe())
print(struct_df.info())

              layer                      name     Position_X    Position_Y  \
0     EMF_em_PCS_05  EMF422-2_string_tilt_15°  692210.844821  735717.65056   
1     EMF_em_PCS_05  EMF422-2_string_tilt_15°  692222.564821  735717.65056   
2     EMF_em_PCS_05  EMF422-2_string_tilt_15°  692234.284821  735717.65056   
3     EMF_em_PCS_05  EMF422-2_string_tilt_15°  692246.004821  735717.65056   
4     EMF_em_PCS_05  EMF422-2_string_tilt_15°  692257.724821  735717.65056   
...             ...                       ...            ...           ...   
4638  EMF_em_PCS_04  EMF422-2_string_tilt_15°  692262.750955  735912.65056   
4639  EMF_em_PCS_04  EMF422-2_string_tilt_15°  692317.505136  735932.15056   
4640  EMF_em_PCS_04  EMF422-2_string_tilt_15°  692381.638455  735727.40056   
4641  EMF_em_PCS_04  EMF422-2_string_tilt_15°  692386.644821  735737.15056   
4642  EMF_em_PCS_04  EMF422-2_string_tilt_15°  692393.358455  735727.40056   

     UFV PST INV EFX STR GRP  
0      0  05  04              
1

Inverter blocks
=
- Keeps the blocks named exactly `EMF422-2_inversor`.
- Renames `X`/`Y` to `inverter_x`/`inverter_y` (so they don't clash with the string positions after the merge) and drops the unused columns.
- `UFV` is zero-padded to 2 digits. The sort (top-to-bottom, left-to-right) only affects the printout.

In [58]:
inverter_mask = (ELM_df['name'] == "EMF422-2_inversor")

inverter_df = ELM_df[inverter_mask].copy()
inverter_df.rename(columns={'X': 'inverter_x', 'Y': 'inverter_y'}, inplace=True)
inverter_df.drop(columns=['name', 'rotation', 'type', 'handle', 'layer', 'EFX', 'STR'], inplace=True)
inverter_df['UFV'] = inverter_df['UFV'].astype(str).str.zfill(2)
inverter_df.sort_values(by=['inverter_y', 'inverter_x'], ascending=[False, True], inplace=True)
print(inverter_df)

         inverter_x     inverter_y UFV PST INV GRP VBT QBT DSJ PVM
4278  692661.355348  736325.370892  00  01  01                    
4279  692724.832637  736296.120892  00  01  02                    
4282  692893.241961  736288.860390  00  01  03                    
4284  693039.806740  736283.518389  00  01  04                    
4280  692762.655229  736276.620892  00  01  05                    
...             ...            ...  ..  ..  ..  ..  ..  ..  ..  ..
4266  695489.680630  734093.985847  00  11  14                    
4253  695258.922630  734084.235847  00  11  15                    
4252  695266.987981  734064.735847  00  11  16                    
4250  695266.987981  734045.227588  00  11  17                    
4251  695323.028091  734025.727588  00  11  18                    

[176 rows x 10 columns]


Building the link key (`GRP`)
=
- `GRP` = `UFV.PST.INV` on both tables (e.g. `01.01.08`). The inverter `INV` goes through `Int64` first so values such as `8` or `08` both become `08`.
- `PST` is used as it is in the attribute, so it must already be 2 digits.
- The two `value_counts()` let you compare the number of strings per inverter and check that each inverter `GRP` appears only once.

In [59]:
struct_df['UFV'] = struct_df["UFV"].astype(str).str.zfill(2)
struct_df['GRP'] = struct_df["UFV"].astype(str) + "." + struct_df["PST"].astype(str) + '.' + struct_df['INV'].astype(str).str.zfill(2)
inverter_df['GRP'] = inverter_df["UFV"].astype(str) + "." + inverter_df["PST"].astype(str) + '.' + inverter_df['INV'].astype('Int64').astype(str).str.zfill(2)

print(struct_df)
print(inverter_df)

              layer                      name     Position_X    Position_Y  \
0     EMF_em_PCS_05  EMF422-2_string_tilt_15°  692210.844821  735717.65056   
1     EMF_em_PCS_05  EMF422-2_string_tilt_15°  692222.564821  735717.65056   
2     EMF_em_PCS_05  EMF422-2_string_tilt_15°  692234.284821  735717.65056   
3     EMF_em_PCS_05  EMF422-2_string_tilt_15°  692246.004821  735717.65056   
4     EMF_em_PCS_05  EMF422-2_string_tilt_15°  692257.724821  735717.65056   
...             ...                       ...            ...           ...   
4638  EMF_em_PCS_04  EMF422-2_string_tilt_15°  692262.750955  735912.65056   
4639  EMF_em_PCS_04  EMF422-2_string_tilt_15°  692317.505136  735932.15056   
4640  EMF_em_PCS_04  EMF422-2_string_tilt_15°  692381.638455  735727.40056   
4641  EMF_em_PCS_04  EMF422-2_string_tilt_15°  692386.644821  735737.15056   
4642  EMF_em_PCS_04  EMF422-2_string_tilt_15°  692393.358455  735727.40056   

     UFV PST INV EFX STR       GRP  
0     00  05  04          

Inverter side and string numbering
=
- `Comparing X` = mean X of the strings of each `GRP` (centroid of the inverter's strings).
- Inner merge with the inverters on `GRP`. Strings without an inverter are dropped silently, and a duplicated inverter `GRP` duplicates strings. Unlike the Reask version there is no row-count check, so compare `len(struct_df)` before and after if in doubt.
- `E or W` = `E` when the inverter is east of (or at) the centroid of its strings, else `W`. It decides the grouping direction later.
- Sort: `UFV`, `PST`, `INV`, then `Position_Y` descending, `Position_X` ascending. `STR` = running number inside each `GRP` (top-to-bottom, left-to-right).
- `TAG` = `GRP.STR` (e.g. `01.01.01.05`), which carries all the tag values into the grouping step.

In [60]:
struct_df['Comparing X'] = struct_df.groupby('GRP')['Position_X'].transform('mean')

struct_df = struct_df.merge(inverter_df, on='GRP')

struct_df['E or W'] = np.where(
    struct_df['inverter_x'] >= struct_df['Comparing X'],
    'E',
    'W'
)

struct_df.sort_values(by=['UFV_x', 'PST_x', 'INV_x', 'Position_Y', 'Position_X'], ascending=[True, True, True, False, True], inplace=True)

struct_df["STR"] = struct_df.groupby("GRP").cumcount()+1
struct_df["TAG"] = struct_df['GRP'] + "." + struct_df['STR'].astype(str).str.zfill(2)


print(struct_df)
print(struct_df["STR"])

              layer                      name     Position_X     Position_Y  \
1925  EMF_em_PCS_01  EMF422-2_string_tilt_15°  692586.996348  736341.905321   
1926  EMF_em_PCS_01  EMF422-2_string_tilt_15°  692598.716348  736341.905321   
1891  EMF_em_PCS_01  EMF422-2_string_tilt_15°  692586.996348  736332.155321   
1892  EMF_em_PCS_01  EMF422-2_string_tilt_15°  692598.716348  736332.155321   
1893  EMF_em_PCS_01  EMF422-2_string_tilt_15°  692610.436348  736332.155321   
...             ...                       ...            ...            ...   
4200  EMF_em_PCS_11  EMF422-2_string_tilt_15°  695409.107091  734022.762016   
3927  EMF_em_PCS_11  EMF422-2_string_tilt_15°  695373.947091  734013.012016   
3926  EMF_em_PCS_11  EMF422-2_string_tilt_15°  695385.667091  734013.012016   
4203  EMF_em_PCS_11  EMF422-2_string_tilt_15°  695397.387091  734013.012016   
4202  EMF_em_PCS_11  EMF422-2_string_tilt_15°  695409.107091  734013.012016   

     UFV_x PST_x INV_x EFX  STR       GRP  ...     

Grouping functions
=
`group_structures(df, inverter_position, max_distance=12)`:
- Sorts by `Position_Y` descending, `Position_X` ascending, and computes the X/Y differences to the previous string (rounded to 2 decimals).
- Starts a new run when `|y_diff| > 0.1` (a new table row), when `|x_diff| > 12` (a gap in the row), or on the first row. There is no `GRP` boundary, so a trio can mix strings of different inverters or layers (the script handles the layer change).
- Cuts each run into trios. Leftovers: 1 string → single, 2 strings → pair.
- For `W` (inverter west of its strings) each run is reversed before cutting, and the list of groups is reversed at the end. This way the trios always start at the end away from the inverter, and the leftover pair/single is on the inverter side.
- Debug side effect: writes `C:\Users\Usuario\Desktop\teste1.xlsx` (overwritten by the second call).

`create_group_positions(groups, inverter_position)`: one row per group with `TAG-1..3` and `Layer-1..3` in west-to-east order (the `W` order is undone here), and the reference position:
- trio: the middle string;
- pair: the western string;
- single: the string itself.

In [61]:
def group_structures(df, inverter_position, max_distance=12):
    """
    Groups adjacent structures in trios (3) while respecting:
    - TAG boundaries
    - Maximum distance constraint (12m default)
    - Sorted by Position Y then X
    """
    # Sort by Position X (primary) and Position Y (secondary)

    df = df.sort_values(['Position_Y', 'Position_X'], ascending=[False, True]).reset_index(drop=True)

    # Calculate pairwise distances between consecutive structures
    df['x_diff'] = round(df['Position_X'].diff(),2)
    df['y_diff'] = round(df['Position_Y'].diff(),2)
    df['distance'] = np.sqrt(df['x_diff']**2 + df['y_diff']**2)

    # Create group boundaries where:
    # 1. Y is different OR
    # 2. Distance > max_distance OR
    # 3. It's the first structure
    df['group_boundary'] = (
        (abs(df['y_diff']) > 0.1) |
        (abs(df['x_diff']) > max_distance) |
        (df.index == 0)
    )
    # Create group IDs
    df['group_id'] = df['group_boundary'].cumsum()

    # Split into groups
    result_groups = []

    for _, group_df in df.groupby('group_id'):
        if inverter_position == "W":
            group_df = group_df.iloc[::-1].reset_index(drop=True)
        # Split groups larger than 3 into trios
        size = len(group_df)
        if size <= 3:
            result_groups.append(group_df)

            continue

        n_trios = size // 3
        for i in range(n_trios):
            result_groups.append(group_df.iloc[i*3:i*3+3])


        if size % 3:  # Handle groups that have sizes that are not divisible by 3
            if (size % 3) % 2:
                result_groups.append(group_df.iloc[-1:])
            else:
                result_groups.append(group_df.iloc[-2:])

    if inverter_position == "W":
        result_groups = result_groups[::-1]
    df.to_excel(r"C:\Users\Usuario\Desktop\teste1.xlsx")
    return result_groups

def create_group_positions(groups, inverter_position):
    """Create the output DataFrame from groups"""
    group_data = []
    for i, group_df in enumerate(groups, 1):
        if len(group_df) == 3:
            if inverter_position == "W":
                first_struct = group_df.iloc[2]
                second_struct = group_df.iloc[1]
                third_struct = group_df.iloc[0]
            else:
                first_struct = group_df.iloc[0]
                second_struct = group_df.iloc[1]
                third_struct = group_df.iloc[2]
            group_data.append({
                'Group_ID': i,
                'TAG-1': first_struct['TAG'],
                'TAG-2': second_struct['TAG'],
                'TAG-3': third_struct['TAG'],
                'Position_X': second_struct['Position_X'],
                'Position_Y': second_struct['Position_Y'],
                'Group_Size': len(group_df),
                'Layer-1': first_struct['layer'],
                'Layer-2': second_struct['layer'],
                'Layer-3': third_struct['layer']
            })
        elif len(group_df) == 2:
            if inverter_position == "W":
                first_struct = group_df.iloc[1]
                second_struct = group_df.iloc[0]
            else:
                first_struct = group_df.iloc[0]
                second_struct = group_df.iloc[1]
            group_data.append({
                'Group_ID': i,
                'TAG-1': first_struct['TAG'],
                'TAG-2': second_struct['TAG'],
                'Position_X': first_struct['Position_X'],
                'Position_Y': first_struct['Position_Y'],
                'Group_Size': len(group_df),
                'Layer-1': first_struct['layer'],
                'Layer-2': second_struct['layer']
            })
        else:
            first_struct = group_df.iloc[0]
            group_data.append({
                'Group_ID': i,
                'TAG-1': first_struct['TAG'],
                'Position_X': first_struct['Position_X'],
                'Position_Y': first_struct['Position_Y'],
                'Group_Size': len(group_df),
                'Layer-1': first_struct['layer']
            })
    return pd.DataFrame(group_data)

Building the group table
=
- Splits the strings into `E` and `W`, groups each side, and concatenates the two group tables. The debug prints (`results_E/W`, columns, dtypes, ...) are active in this version.
- Reads the tag values back from `TAG-1/2/3` by fixed slices: `UFV` `[0:2]`, `PST` `[3:5]`, `INV` `[6:8]`, `STR` last 2 characters, `GRP` = everything but `.STR`. All four parts must be exactly 2 digits.
- The `-2`/`-3` columns are `None` when the group is smaller.
- Writes a debug copy to `C:\Users\Usuario\Desktop\teste.xlsx`.

In [62]:
estruturas_E = struct_df[struct_df['E or W'] == 'E']
estruturas_W = struct_df[struct_df['E or W'] == 'W']

results_E = group_structures(estruturas_E, 'E')
results_W = group_structures(estruturas_W, 'W')
print(results_E)
print(results_W)

group_positions_E = create_group_positions(results_E, "E")
group_positions_W = create_group_positions(results_W, "W")

group_positions = pd.concat([group_positions_E, group_positions_W], ignore_index=True)

# DEBUG: Check what columns actually exist
print("Columns in group_positions after concat:")
print(group_positions.columns.tolist())
print("\nFirst 5 rows:")
print(group_positions.head())
print("\nData types:")
print(group_positions.dtypes)
print("\nUnique Group_Size values:")
print(group_positions['Group_Size'].unique())

group_positions["UFV-1"] = group_positions['TAG-1'].str[0:2]
group_positions['UFV-2'] = np.where(
    group_positions["Group_Size"] > 1,
    group_positions['TAG-2'].str[0:2],
    None
)
group_positions['UFV-3'] = np.where(
    group_positions["Group_Size"] > 2,
    group_positions['TAG-3'].str[0:2],
    None
)

group_positions["PST-1"] = group_positions['TAG-1'].str[3:5]
group_positions['PST-2'] = np.where(
    group_positions["Group_Size"] > 1,
    group_positions['TAG-2'].str[3:5],
    None
)
group_positions['PST-3'] = np.where(
    group_positions["Group_Size"] > 2,
    group_positions['TAG-3'].str[3:5],
    None
)

group_positions["INV-1"] = group_positions['TAG-1'].str[6:8]
group_positions['INV-2'] = np.where(
    group_positions["Group_Size"] > 1,
    group_positions['TAG-2'].str[6:8],
    None
)
group_positions['INV-3'] = np.where(
    group_positions["Group_Size"] > 2,
    group_positions['TAG-3'].str[6:8],
    None
)

group_positions["STR-1"] = group_positions['TAG-1'].str[-2:]
group_positions['STR-2'] = np.where(
    group_positions["Group_Size"] > 1,
    group_positions['TAG-2'].str[-2:],
    None
)
group_positions['STR-3'] = np.where(
    group_positions["Group_Size"] > 2,
    group_positions['TAG-3'].str[-2:],
    None
)

group_positions['GRP-1'] = group_positions['TAG-1'].str[:-3]
group_positions['GRP-2'] = np.where(
    group_positions["Group_Size"] > 1,
    group_positions['TAG-2'].str[:-3],
    None
)
group_positions['GRP-3'] = np.where(
    group_positions["Group_Size"] > 2,
    group_positions['TAG-3'].str[:-3],
    None
)

print(group_positions)
group_positions.to_excel(r"C:\Users\Usuario\Desktop\teste.xlsx")

[           layer                      name     Position_X     Position_Y  \
0  EMF_em_PCS_01  EMF422-2_string_tilt_15°  692586.996348  736341.905321   
1  EMF_em_PCS_01  EMF422-2_string_tilt_15°  692598.716348  736341.905321   

  UFV_x PST_x INV_x EFX  STR       GRP  ...  QBT  DSJ  PVM E or W  \
0    00    01    01        1  00.01.01  ...                     E   
1    00    01    01        2  00.01.01  ...                     E   

           TAG x_diff y_diff distance group_boundary group_id  
0  00.01.01.01    NaN    NaN      NaN           True        1  
1  00.01.01.02  11.72    0.0    11.72          False        1  

[2 rows x 27 columns],            layer                      name     Position_X     Position_Y  \
2  EMF_em_PCS_01  EMF422-2_string_tilt_15°  692586.996348  736332.155321   
3  EMF_em_PCS_01  EMF422-2_string_tilt_15°  692598.716348  736332.155321   
4  EMF_em_PCS_01  EMF422-2_string_tilt_15°  692610.436348  736332.155321   

  UFV_x PST_x INV_x EFX  STR       GRP  .

Script generation
=
`generate_script(row)` builds the commands for one group (each block gets `UFV`, `PST`, `INV`, `""`, `STR`, `GRP`, scale `1 1`):
- **Single:** `EMF422-1_string_1` at the string position, rotation 0.
- **Pair:** two `EMF422-1_string_2` at the same point (western string X + `5.86`, half the 11.72 m pitch, i.e. the middle of the pair), rotations 0 and 180.
- **Trio:** three `EMF422-1_string_3` at the middle string X, with Y + `2.3204`, Y and Y - `2.3204`, rotation 0.
- `._layer set` is written before the first block and again whenever the layer changes inside a group.
- **Warning:** the block names are still the Reask ones (`EMF422-1_...`), while the source blocks here are `EMF422-2_...`. Check which string blocks exist in the Toolestown drawing.

Output: `Separador de Strings - Toolestown.scr`, plus a copy of the table in `teste.xlsx` in the same folder.

In [63]:
correction_position_x_2str = 5.86
correction_position_y_3str = 2.3204

def generate_script(row):
    group_size = row['Group_Size']

    if group_size == 1:
        return (
            f'(command-s "._layer" "set" "{row["Layer-1"]}" "")\n'
            f'(command-s "._insert" "EMF422-1_string_{group_size}" '
            f'"{round(row["Position_X"],4)},{round(row["Position_Y"],4)}" 1 1 0 "{row["UFV-1"]}" "{row["PST-1"]}" "{row["INV-1"]}" "" "{row["STR-1"]}" "{row["GRP-1"]}")'
        )

    elif group_size == 2:
        if row["Layer-1"] == row["Layer-2"]:
            return (
                f'(command-s "._layer" "set" "{row["Layer-1"]}" "")\n'
                f'(command-s "._insert" "EMF422-1_string_{group_size}" '
                f'"{round(row["Position_X"]+correction_position_x_2str,4)},{round(row["Position_Y"],4)}" 1 1 0 "{row["UFV-1"]}" "{row["PST-1"]}" "{row["INV-1"]}" "" "{row["STR-1"]}" "{row["GRP-1"]}")\n'
                f'(command-s "._insert" "EMF422-1_string_{group_size}" '
                f'"{round(row["Position_X"]+correction_position_x_2str,4)},{round(row["Position_Y"],4)}" 1 1 180 "{row["UFV-2"]}" "{row["PST-2"]}" "{row["INV-2"]}" "" "{row["STR-2"]}" "{row["GRP-2"]}")'
            )
        else:
            return (
                f'(command-s "._layer" "set" "{row["Layer-1"]}" "")\n'
                f'(command-s "._insert" "EMF422-1_string_{group_size}" '
                f'"{round(row["Position_X"]+correction_position_x_2str,4)},{round(row["Position_Y"],4)}" 1 1 0 "{row["UFV-1"]}" "{row["PST-1"]}" "{row["INV-1"]}" "" "{row["STR-1"]}" "{row["GRP-1"]}")\n'
                f'(command-s "._layer" "set" "{row["Layer-2"]}" "")\n'
                f'(command-s "._insert" "EMF422-1_string_{group_size}" '
                f'"{round(row["Position_X"]+correction_position_x_2str,4)},{round(row["Position_Y"],4)}" 1 1 180 "{row["UFV-2"]}" "{row["PST-2"]}" "{row["INV-2"]}" "" "{row["STR-2"]}" "{row["GRP-2"]}")'
            )

    else:  # group_size == 3
        # Check if all layers are the same
        if row["Layer-1"] == row["Layer-2"] == row["Layer-3"]:
            return (
                f'(command-s "._layer" "set" "{row["Layer-1"]}" "")\n'
                f'(command-s "._insert" "EMF422-1_string_{group_size}" '
                f'"{round(row["Position_X"],4)},{round(row["Position_Y"]+correction_position_y_3str,4)}" 1 1 0 "{row["UFV-1"]}" "{row["PST-1"]}" "{row["INV-1"]}" "" "{row["STR-1"]}" "{row["GRP-1"]}")\n'
                f'(command-s "._insert" "EMF422-1_string_{group_size}" '
                f'"{round(row["Position_X"],4)},{round(row["Position_Y"],4)}" 1 1 0 "{row["UFV-2"]}" "{row["PST-2"]}" "{row["INV-2"]}" "" "{row["STR-2"]}" "{row["GRP-2"]}")\n'
                f'(command-s "._insert" "EMF422-1_string_{group_size}" '
                f'"{round(row["Position_X"],4)},{round(row["Position_Y"]-correction_position_y_3str,4)}" 1 1 0 "{row["UFV-3"]}" "{row["PST-3"]}" "{row["INV-3"]}" "" "{row["STR-3"]}" "{row["GRP-3"]}")'
            )
        else:
            script = f'(command-s "._layer" "set" "{row["Layer-1"]}" "")\n'
            script += f'(command-s "._insert" "EMF422-1_string_{group_size}" '
            script += f'"{round(row["Position_X"],4)},{round(row["Position_Y"]+correction_position_y_3str,4)}" 1 1 0 "{row["UFV-1"]}" "{row["PST-1"]}" "{row["INV-1"]}" "" "{row["STR-1"]}" "{row["GRP-1"]}")\n'

            if row["Layer-1"] != row["Layer-2"]:
                script += f'(command-s "._layer" "set" "{row["Layer-2"]}" "")\n'

            script += f'(command-s "._insert" "EMF422-1_string_{group_size}" '
            script += f'"{round(row["Position_X"],4)},{round(row["Position_Y"],4)}" 1 1 0 "{row["UFV-2"]}" "{row["PST-2"]}" "{row["INV-2"]}" "" "{row["STR-2"]}" "{row["GRP-2"]}")\n'

            if row["Layer-2"] != row["Layer-3"]:
                script += f'(command-s "._layer" "set" "{row["Layer-3"]}" "")\n'

            script += f'(command-s "._insert" "EMF422-1_string_{group_size}" '
            script += f'"{round(row["Position_X"],4)},{round(row["Position_Y"]-correction_position_y_3str,4)}" 1 1 0 "{row["UFV-3"]}" "{row["PST-3"]}" "{row["INV-3"]}" "" "{row["STR-3"]}" "{row["GRP-3"]}")'

            return script

# Apply the function to each row
group_positions['Script'] = group_positions.apply(generate_script, axis=1)

print(group_positions)

with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Separador de Strings\Separador de Strings - Toolestown.scr", 'w') as f:
    f.write('\n'.join(group_positions["Script"].astype(str)) + '\n')

group_positions.to_excel(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Separador de Strings\teste.xlsx")

      Group_ID        TAG-1        TAG-2     Position_X     Position_Y  \
0            1  00.01.01.01  00.01.01.02  692586.996348  736341.905321   
1            2  00.01.01.03  00.01.01.04  692598.716348  736332.155321   
2            3  00.01.01.06          NaN  692622.156348  736332.155321   
3            4  00.01.01.07  00.01.01.08  692598.716348  736322.405321   
4            5  00.01.01.10  00.01.01.11  692633.876348  736322.405321   
...        ...          ...          ...            ...            ...   
1675       878  00.01.03.09  00.01.03.10  692919.991562  736305.394819   
1676       879  00.01.03.07  00.01.03.08  692919.991562  736315.144819   
1677       880  00.01.03.05  00.01.03.06  692919.991562  736324.894819   
1678       881  00.01.03.03  00.01.03.04  692925.851562  736334.644819   
1679       882  00.01.03.01  00.01.03.02  692925.851562  736344.394819   

      Group_Size        Layer-1        Layer-2        TAG-3        Layer-3  \
0              2  EMF_em_PCS_01  